# From shadows to observation kernels

This lesson climbs five levels: a visual story, a matrix model, a nonlinear verdict, a singular-value bound, and a proof-level interpretation. Run the cells in order.

## Level 1 — intuition: a shadow forgets motion

A camera that records only horizontal position cannot see vertical motion. Changing the software that labels the same photograph does not restore the missing coordinate.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import FloatSlider, interact

PAPER = Path.cwd()
sys.path.insert(0, str(PAPER / 'code'))
from lggr_vi_core import ObservationSystem, unanimous_green, k_of_n_green

system = ObservationSystem([[1.0, 0.0]])
system.observe([0.0, 2.0])

In [ ]:
def plot_state(visible, hidden):
    state = np.array([visible, hidden])
    report = system.observe(state)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.axvline(0, color='tab:blue', linewidth=5, alpha=0.25, label='kernel')
    ax.arrow(0, 0, *state, width=0.02, color='tab:red', length_includes_head=True)
    ax.set(xlim=(-2.2, 2.2), ylim=(-2.2, 2.2), title=f'Qx = {report[0]:.2f}')
    ax.grid(alpha=0.2); ax.legend(); display(fig); plt.close(fig)
    return {'state': state.tolist(), 'report': report.tolist()}

@interact(
    visible=FloatSlider(min=-2, max=2, step=0.1, value=0.5),
    hidden=FloatSlider(min=-2, max=2, step=0.1, value=1.0),
)
def explore_state(visible, hidden):
    return plot_state(visible, hidden)

### Static fallback

The representative state below uses the same plotting function and remains visible when interactive widgets are unavailable.

In [ ]:
static_state = plot_state(visible=0.5, hidden=1.0)
assert static_state['report'] == [0.5]
static_state

## Level 2 — the operator

For a matrix $Q$, two states are observationally equivalent when $Qx=Qy$. Their difference lies in $\ker Q$. In this example the kernel is the vertical axis.

In [ ]:
kernel = system.kernel_basis()
visible = system.visible_basis()
assert system.is_in_kernel(kernel[:, 0])
{'kernel_basis': kernel, 'visible_basis': visible, 'rank': system.rank()}

## Level 3 — nonlinear verdicts

Suppose every local score is zero. Unanimous and $k$-of-$n$ rules both accept. The important premise is not that the rules are linear; it is that they receive only the report.

In [ ]:
scores = [0.0, 0.0, 0.0]
assert unanimous_green(scores)
assert k_of_n_green(scores, k=2)
{'unanimous': unanimous_green(scores), 'two_of_three': k_of_n_green(scores, 2)}

## Level 4 — the singular-value cylinder

If $\|Qx\|_2\leq\tau$, then the visible distance from $x$ to $\ker Q$ is at most $\tau/\sigma_{\min}^{+}(Q)$. Hidden motion is unbounded; only the visible component is controlled.

In [ ]:
scaled = ObservationSystem([[3.0, 0.0]])
state = np.array([0.2, 100.0])
tau = np.linalg.norm(scaled.observe(state))
bound = scaled.cylinder_radius(tau)
distance = scaled.distance_to_kernel(state)
assert distance <= bound + 1e-12
{'observed_norm': tau, 'distance_to_kernel': distance, 'bound': bound}

## Level 5 — proof and research questions

Write $x=k+u$ with $k\in\ker Q$ and $u\perp\ker Q$. Then $Qx=Qu$ and $\|Qu\|_2\geq\sigma_{\min}^{+}\|u\|_2$. This proves the cylinder bound.

Questions for further work: What changes if the observation itself is adaptive? How should a tolerance depend on measurement noise? Which added row of $Q$ reduces the kernel most efficiently?